# 02 - Model A: Standard CNN

Build, train, and evaluate **Model A** (a standard CNN: interleaved `Conv2D` + `MaxPooling2D` layers) on the same Intel Image Classification splits cached by `01_data_preparation.ipynb` (teammate 1), and save results using the same Drive-based conventions as `06_sota_mobilenet.ipynb` so everything can be compared directly in `08_final_comparison.ipynb`.

In [ ]:
# --- Environment setup (Google Colab + shared Google Drive, with a local fallback) ---
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the team's shared Drive - this is where 01_data_preparation.ipynb
    # cached the resized/normalized dataset, and where we save checkpoints +
    # results so nobody has to download large files to their own PC.
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/EN3150-A03-Resource-Constrained-CNN"
else:
    # Not in Colab (e.g. running locally to sanity-check the code) - use
    # this repo's own local folders instead of Drive.
    REPO_DIR = os.path.abspath("..")

print("IN_COLAB:", IN_COLAB)
print("REPO_DIR:", REPO_DIR)


## 1. Config

In [ ]:
# --- Config for this notebook (defined here, not in a shared file) ---
CONFIG = {
    "image_size": 64,          # resolution of the cached dataset (<= 64x64 per spec)
    "channels": 3,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "random_seed": 42,
    "batch_size": 32,
    "epochs": 20,               # assignment requires >= 20 epochs
    "learning_rate": 0.001,     # Adam - compared against SGD / SGD+momentum in 04_optimizer_comparison.ipynb
    "dataset_name": "Intel Image Classification",
    "num_classes": 6,
    "class_names": ["buildings", "forest", "glacier", "mountain", "sea", "street"],
}


# --- Paths (mirrors the layout used by 01_data_preparation.ipynb / 06_sota_mobilenet.ipynb) ---
PROCESSED_DIR = os.path.join(REPO_DIR, "data", "processed", "intel-image-classification")
CHECKPOINT_DIR = os.path.join(REPO_DIR, "models", "checkpoints", "model_a")
FINAL_MODEL_DIR = os.path.join(REPO_DIR, "models", "final")
FIGURES_DIR = os.path.join(REPO_DIR, "results", "figures", "model_a")
CONFUSION_DIR = os.path.join(REPO_DIR, "results", "confusion_matrices")
METRICS_DIR = os.path.join(REPO_DIR, "results", "metrics")

for d in (CHECKPOINT_DIR, FINAL_MODEL_DIR, FIGURES_DIR, CONFUSION_DIR, METRICS_DIR):
    os.makedirs(d, exist_ok=True)

print("PROCESSED_DIR:", PROCESSED_DIR)
CONFIG


## 2. Imports

In [ ]:
import json
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 3. Seed

In [ ]:
SEED = CONFIG["random_seed"]
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 4. Load Prepared Data

In [ ]:
# Load the same 64x64 float32 [0, 1] splits produced by 01_data_preparation.ipynb.
X_train = np.load(os.path.join(PROCESSED_DIR, "X_train.npy"))
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"))
X_val = np.load(os.path.join(PROCESSED_DIR, "X_val.npy"))
y_val = np.load(os.path.join(PROCESSED_DIR, "y_val.npy"))
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

with open(os.path.join(PROCESSED_DIR, "class_names.json")) as f:
    class_names = json.load(f)
assert class_names == CONFIG["class_names"], "class_names.json does not match CONFIG - re-check notebook 01"

print("train/val/test sizes:", len(X_train), len(X_val), len(X_test))
print("image shape:", X_train.shape[1:], "dtype:", X_train.dtype)


## 5. Build Model A

In [ ]:
def build_model_a(input_shape, num_classes):
    """Model A: standard CNN - interleaved Conv2D and MaxPooling2D layers,
    ending in a small dense classification head (see Figure 1 in the brief)."""
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),

        keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),

        keras.layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),

        keras.layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),

        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes, activation="softmax"),
    ], name="model_a_standard_cnn")
    return model


input_shape = (CONFIG["image_size"], CONFIG["image_size"], CONFIG["channels"])
model = build_model_a(input_shape, CONFIG["num_classes"])


## 6. Model Summary

In [ ]:
model.summary()


## 7. Parameter Count

In [ ]:
total_params = model.count_params()
print(f"Total trainable parameters: {total_params:,}")


## 8. Activation Function Justification

**ReLU** (`max(0, x)`) is used in every hidden `Conv2D`/`Dense` layer, and **softmax** only at the output layer (required to produce a multi-class probability distribution).

- ReLU is a single comparison-and-clamp operation - no exponential - so it is far cheaper to evaluate on constrained/edge hardware than `sigmoid`/`tanh`, both of which require an exponential per activation, across every pixel of every feature map.
- ReLU also avoids the vanishing-gradient problem `sigmoid`/`tanh` suffer from as networks get deeper, which matters here since Model A stacks three convolutional stages.
- Softmax's exponential cost is negligible in comparison: it is evaluated once per forward pass over only `num_classes` (6) values, not per feature-map pixel like the hidden-layer activations.

## 9. Training (>= 20 epochs)

In [ ]:
ckpt_path = os.path.join(CHECKPOINT_DIR, "best.keras")

checkpoint_cb = keras.callbacks.ModelCheckpoint(
    ckpt_path, monitor="val_loss", save_best_only=True, verbose=1,
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

t0 = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    shuffle=True,
    callbacks=[checkpoint_cb],
)
total_time = time.time() - t0
avg_time_per_epoch = total_time / CONFIG["epochs"]
print(f"Training ({CONFIG['epochs']} epochs) took {total_time:.1f}s ({avg_time_per_epoch:.1f}s/epoch)")

# Reload the best checkpoint (lowest val_loss, not necessarily the last epoch) before evaluating.
model = keras.models.load_model(ckpt_path)


## 10. Training / Validation Loss Curves

In [ ]:
epochs_range = range(1, CONFIG["epochs"] + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs_range, history.history["loss"], label="train")
axes[0].plot(epochs_range, history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs_range, history.history["accuracy"], label="train")
axes[1].plot(epochs_range, history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.suptitle("Model A - Standard CNN")
plt.tight_layout()
loss_curve_path = os.path.join(FIGURES_DIR, "loss_curve.png")
plt.savefig(loss_curve_path, dpi=150)
plt.show()
print("Saved:", loss_curve_path)


## 11. Test Set Evaluation

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision_macro = precision_score(y_test, y_pred, average="macro")
test_recall_macro = recall_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print(f"Test accuracy:           {test_accuracy:.4f}")
print(f"Test precision (macro):  {test_precision_macro:.4f}")
print(f"Test recall (macro):     {test_recall_macro:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=class_names))


## 12. Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Model A - Confusion Matrix (test set)")
plt.tight_layout()
cm_path = os.path.join(CONFUSION_DIR, "model_a.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print("Saved:", cm_path)


## 13. Model Size & Training Time

In [ ]:
final_model_path = os.path.join(FINAL_MODEL_DIR, "model_a.keras")
model.save(final_model_path)

model_size_kb = os.path.getsize(final_model_path) / 1024
print(f"Saved final model to: {final_model_path}")
print(f"Model size on disk: {model_size_kb:.1f} KB")
print(f"Average training time per epoch: {avg_time_per_epoch:.1f}s")


## 14. Save Results

In [ ]:
metrics = {
    "model_name": "Model A (Standard CNN)",
    "dataset_name": CONFIG["dataset_name"],
    "test_accuracy": float(test_accuracy),
    "test_precision_macro": float(test_precision_macro),
    "test_recall_macro": float(test_recall_macro),
    "confusion_matrix": cm.tolist(),
    "class_names": class_names,
    "total_params": int(total_params),
    "model_size_kb": float(model_size_kb),
    "epochs": CONFIG["epochs"],
    "avg_time_per_epoch_sec": float(avg_time_per_epoch),
    "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
}

metrics_path = os.path.join(METRICS_DIR, "model_a.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics to:", metrics_path)


## 15. Conclusion

In [ ]:
print(
    f"Model A (Standard CNN, {CONFIG['image_size']}x{CONFIG['image_size']} input):\n"
    f"  Test accuracy:                  {test_accuracy:.4f}\n"
    f"  Test precision/recall (macro):  {test_precision_macro:.4f} / {test_recall_macro:.4f}\n"
    f"  Total parameters:               {total_params:,}\n"
    f"  Model size on disk:             {model_size_kb:.1f} KB\n"
    f"  Avg. time/epoch:                {avg_time_per_epoch:.1f}s"
)


TODO (fill in after running on the actual dataset/runtime, using the numbers printed above): summarize Model A's performance and any qualitative observations from the confusion matrix (e.g. which scene classes are most often confused, such as `glacier` vs. `mountain`, or `sea` vs. `buildings` for coastal cityscapes). Carry these numbers into `05_custom_models_evaluation.ipynb` and `08_final_comparison.ipynb`.